# 04. Chuẩn bị dữ liệu (Data Preparation)

Đầu vào: `data/processed/products.parquet` (notebook 02). Mỗi vấn đề ở cuối notebook 03 được xử lý ở đây:

| Mục | Việc làm | Mã |
|---|---|---|
| 6.1 | Chuyển kiểu dữ liệu, chuẩn hóa thương hiệu/thẻ, xóa thẻ lộ đáp án | `cleaning.normalize` |
| 6.2 | Lọc dòng lỗi, ngoại lai phi lý, loại trùng | `cleaning.apply_filters` |
| 6.3 | Xử lý giá trị thiếu, gắn cờ chất lượng, tạo nhóm thương hiệu | `cleaning.add_flags_and_groups` |
| 6.4 | Tách X / y, kiểm tra chống rò rỉ, lưu tệp | `cleaning.clean` |
| 6.5 | Chia Train / Validation / Test theo thương hiệu, khóa test | `modeling.make_split` |
| 6.6 | Feature engineering, encoding, chuẩn hóa, chọn đặc trưng | `features.make_features` |

Nguyên tắc: **không lọc dòng dựa trên bảng dinh dưỡng** (dự án mô phỏng sản phẩm thiếu dinh dưỡng; lọc theo dinh dưỡng làm lệch dữ liệu).

In [1]:
# Chạy từ thư mục gốc dự án để đọc được data/, sql/ và import src/
import os, sys
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

from src import cleaning
products = pd.read_parquet("data/processed/products.parquet")
EXAMPLE = "3457221000505"          # Caramel au beurre salé, dùng làm ví dụ xuyên suốt
products.shape

(30000, 29)

## 6.1. Chuyển kiểu và chuẩn hóa
- Thương hiệu: `xx:Carrefour`, `Carrefour `, `carrefour` là một → bỏ tiền tố ngôn ngữ, bỏ dấu, chữ thường (`slugify`).
- Danh sách thẻ: chữ thường, bỏ khoảng trắng, bỏ thẻ trùng; quốc gia bỏ `en:world`.
- **Xóa thẻ lộ đáp án** (logo Nutri-Score, Health Star, "exceso"...) khỏi `labels_tags`, `categories_tags`, giữ nguyên sản phẩm.
- Số đếm: số âm là lỗi → thiếu; đổi sang kiểu số nguyên cho phép thiếu (`Int64`). Mã vạch giữ kiểu chuỗi.

In [2]:
norm = cleaning.normalize(products)

pd.DataFrame({"brands_tags (trước)": products["brands_tags"].head(6),
              "brands_tags (sau)": norm["brands_tags"].head(6),
              "brand": norm["brand"].head(6)})

,brands_tags (trước),brands_tags (sau),brand
0,[xx:Smuckers Uncrustables],[smuckers-uncrustables],smuckers-uncrustables
1,[xx:volg],[volg],volg
2,[xx:Casino],[casino],casino
3,[pams],[pams],pams
4,[xx:Quokka],[quokka],quokka
5,[xx:rsp],[rsp],rsp


In [3]:
first_brand = products["brands_tags"].map(lambda b: b[0] if b is not None and len(b) else None)
print("Số thương hiệu khác nhau: trước", first_brand.nunique(), "-> sau", norm["brand"].nunique())
print("Dòng có en:world (đã bỏ thẻ):", int(products["countries_tags"].map(lambda t: t is not None and "en:world" in t).sum()))
for col in cleaning.LEAKY_TAG_COLUMNS:
    n = products[col].map(lambda t: any(cleaning.is_leaky_label(x) for x in (t if t is not None else []))).sum()
    print(f"Dòng có thẻ lộ đáp án trong {col} (đã xóa thẻ, giữ dòng): {n}")

pd.DataFrame({"trước": products[["code", "ingredients_n", "nova_group", "last_modified"]].dtypes.astype(str),
              "sau": norm[["code", "ingredients_n", "nova_group", "last_modified"]].dtypes.astype(str)})

Số thương hiệu khác nhau: trước 12940 -> sau 10835


Dòng có en:world (đã bỏ thẻ): 709


Dòng có thẻ lộ đáp án trong labels_tags (đã xóa thẻ, giữ dòng): 2263


Dòng có thẻ lộ đáp án trong categories_tags (đã xóa thẻ, giữ dòng): 1


,trước,sau
code,str,str
ingredients_n,float64,Int64
nova_group,float64,Int64
last_modified,"datetime64[us, UTC+07:00]","datetime64[us, UTC]"


## 6.2. Lọc, xử lý ngoại lai, loại trùng (bảng phễu)
Mỗi bước ghi số dòng bị loại. Thứ tự ưu tiên khi trùng mã vạch: giữ bản sửa gần nhất, để kết quả không phụ thuộc thứ tự đọc.

Ngoại lai: chỉ loại giá trị **phi lý** (> 150 thành phần). Sản phẩm nhiều thành phần theo quy tắc IQR là có thật nên giữ; sản phẩm ≤ 1 thành phần (mật ong, dầu ô liu) cũng giữ, chỉ gắn cờ.

In [4]:
filtered, funnel = cleaning.apply_filters(norm)
funnel

,Bước,Loại bỏ,Còn lại
0,Đầu vào,0,30000
1,Nhãn không phải A-E,2,29998
2,Không có danh sách thành phần,0,29998
3,Mã vạch thiếu hoặc có ký tự không phải số,0,29998
4,Số thành phần > 150,19,29979
5,Trùng mã vạch,0,29979
6,Trùng nội dung (thương hiệu + tên + thành phần),75,29904


## 6.3. Giá trị thiếu, cờ chất lượng và nhóm thương hiệu

| Cột thiếu | Cách xử lý | Vì sao |
|---|---|---|
| `pnns_groups_1/2` (~20%) | giá trị riêng `"missing"` khi mã hóa | thiếu nhóm cũng là thông tin (EDA B11: nhóm thiếu khó đoán hơn hẳn) |
| `nova_group` (~10%) | giá trị riêng `"missing"` | như trên; không đoán bừa một nhóm NOVA |
| Số đếm (`ingredients_n`, ...) | điền **trung vị của tập huấn luyện** (trong pipeline đặc trưng) | không lấy thông tin từ tập kiểm tra |
| Danh sách thẻ rỗng | để rỗng (mọi cột multi-hot = 0) | "không có phụ gia" là thông tin thật |
| Thương hiệu (~15%) | nhóm theo tên sản phẩm đã chuẩn hóa | không gộp mọi sản phẩm thiếu thương hiệu thành một nhóm khổng lồ |

Cờ chất lượng chỉ để theo dõi, **không** loại dòng.

In [5]:
final = cleaning.add_flags_and_groups(filtered)
flags = ["flag_few_ingredients", "flag_missing_nova", "flag_missing_brand"]
print((final[flags].mean() * 100).round(1).to_dict())
print("Có ít nhất một thẻ thành phần ngoài taxonomy:", f"{(final['share_ingredients_off_taxonomy'] > 0).mean():.1%}")

groups = final["group_id"].value_counts()
print("Số nhóm:", len(groups), "| nhóm lớn nhất:", groups.index[0], groups.iloc[0], f"dòng ({groups.iloc[0] / len(final):.1%})")
final["group_id"].str.split(":").str[0].value_counts().rename("loại nhóm").to_frame().T

{'flag_few_ingredients': 13.3, 'flag_missing_nova': 9.7, 'flag_missing_brand': 15.2}
Có ít nhất một thẻ thành phần ngoài taxonomy: 31.0%
Số nhóm: 15141 | nhóm lớn nhất: brand:carrefour 328 dòng (1.1%)


group_id,brand,name,code
loại nhóm,25348,4317,239


## 6.4. Tách X / y, chống rò rỉ, lưu tệp
`cleaning.clean` chạy lần lượt 6.1 → 6.3 rồi tách:
- **X** (`clean_features.parquet`, 27 cột): chỉ đặc trưng hợp lệ + `group_id`, cờ; **không có nhãn, không có dinh dưỡng**.
- **y** (`clean_labels.parquet`): `row_id`, `code`, `nutriscore_grade`.

Hai lớp chặn rò rỉ: `assert_no_leakage` kiểm tra **tên cột**, `assert_no_leaky_values` kiểm tra **giá trị** thẻ.

In [6]:
X, y, funnel = cleaning.clean(products)
assert X["row_id"].equals(final["row_id"])            # cùng kết quả với từng bước ở trên
X.to_parquet("data/final/clean_features.parquet", index=False)
y.to_parquet("data/final/clean_labels.parquet", index=False)
print("X:", X.shape, "| y:", y.shape)
print(y["nutriscore_grade"].value_counts().sort_index().to_dict())
list(X.columns)

X: (29904, 27) | y: (29904, 3)
{'a': 4517, 'b': 3296, 'c': 6482, 'd': 7112, 'e': 8497}


['row_id',
 'code',
 'group_id',
 'product_name',
 'brand',
 'brands_tags',
 'countries_tags',
 'n_countries',
 'categories_tags',
 'pnns_groups_1',
 'pnns_groups_2',
 'ingredients_tags',
 'ingredients_text',
 'ingredients_n',
 'ingredients_from_palm_oil_n',
 'ingredients_analysis_tags',
 'additives_tags',
 'additives_n',
 'allergens_tags',
 'labels_tags',
 'nova_group',
 'lang',
 'last_modified',
 'share_ingredients_off_taxonomy',
 'flag_few_ingredients',
 'flag_missing_nova',
 'flag_missing_brand']

In [7]:
cleaning.assert_no_leakage(X.columns)
cleaning.assert_no_leaky_values(X)
print("X sạch: không có cột bị cấm, không còn thẻ lộ đáp án")

# Thử cố tình đưa cột dinh dưỡng vào: phải bị chặn
try:
    cleaning.assert_no_leakage(list(X.columns) + ["sugars_100g"])
except ValueError as e:
    print("Bị chặn:", e)

X sạch: không có cột bị cấm, không còn thẻ lộ đáp án
Bị chặn: Bảng đặc trưng chứa cột bị cấm: ['sugars_100g']


## 6.5. Chia Train / Validation / Test

Vì sao chia **trước** khi tạo đặc trưng? Một số đặc trưng phải "học" từ dữ liệu (thẻ nào đủ phổ biến, trung vị, trung bình và độ lệch chuẩn để chuẩn hóa). Nếu học trên cả tập kiểm tra thì thông tin tập kiểm tra lọt vào mô hình.

Vì sao chia **theo thương hiệu**? Sản phẩm cùng hãng có công thức rất giống nhau. Nếu chia ngẫu nhiên, mô hình gặp "anh em" của sản phẩm test trong lúc học → điểm cao ảo. `StratifiedGroupKFold` giữ mỗi thương hiệu trọn trong một phần, đồng thời giữ tỷ lệ hạng A–E gần như nhau giữa các phần.

| Phần | Tỷ lệ | Dùng để |
|---|---|---|
| **Test** | 20% | đánh giá cuối, **mở đúng một lần** sau khi chốt mô hình (đang khóa) |
| **Dev** = Train + Validation | 80% | chia 5 fold: mỗi lần 4 fold để huấn luyện, 1 fold để kiểm định (đánh giá chéo) |

Tập chia được tạo một lần ở Bước 7 và lưu ở `data/final/split.parquet`. Ô dưới tạo lại bằng mã hiện tại và **so với tệp đã lưu**: phải giống hệt, và dấu vân tay (SHA-256 của danh sách mã vạch test) phải là `2e511eab89b680b7`.

In [8]:
from src.modeling import check_split, fingerprint, make_split, select_dev, select_test

saved = pd.read_parquet("data/final/split.parquet")
new = make_split(X, y["nutriscore_grade"])
same = all((new[c].astype(str).to_numpy() == saved[c].astype(str).to_numpy()).all() for c in saved.columns)
print("Tạo lại giống hệt tệp đã khóa:", same)
print("Vân tay tập test:", fingerprint(new), "| tệp đã lưu:", fingerprint(saved))
print("Không nhóm nào nằm ở hai phần:", check_split(saved))

Tạo lại giống hệt tệp đã khóa: True
Vân tay tập test: 2e511eab89b680b7 | tệp đã lưu: 2e511eab89b680b7


Không nhóm nào nằm ở hai phần: True


In [9]:
part = np.where(saved["split"] == "test", "test", "dev fold " + saved["fold"].astype(str))
dist = pd.crosstab(pd.Series(part, name="phần"), pd.Series(y["nutriscore_grade"].to_numpy(), name="% hạng"),
                   normalize="index").mul(100).round(1)
dist.insert(0, "số dòng", pd.Series(part).value_counts())
dist.insert(1, "số nhóm", saved.groupby(part)["group_id"].nunique())
dist

% hạng,số dòng,số nhóm,a,b,c,d,e
phần,,,,,,,
dev fold 0,4783,2408,15.1,11.0,21.7,23.8,28.4
dev fold 1,4783,2424,15.1,11.0,21.7,23.8,28.4
dev fold 2,4786,2429,15.1,11.0,21.7,23.8,28.4
dev fold 3,4787,2417,15.1,11.0,21.7,23.8,28.4
dev fold 4,4783,2410,15.1,11.0,21.7,23.8,28.4
test,5982,3053,15.1,11.0,21.7,23.8,28.4


Tỷ lệ hạng của mọi phần gần như bằng nhau (lệch dưới 0,5 điểm %). Tập test bị khóa bằng mã: gọi mà không có xác nhận cuối thì báo lỗi.

In [10]:
try:
    select_test(X, y["nutriscore_grade"], saved)
except PermissionError as e:
    print("Bị chặn:", e)

X_dev, y_dev, split_dev = select_dev(X, y["nutriscore_grade"], saved)
print("Dev:", X_dev.shape)

Bị chặn: Tập test bị khóa. Chỉ gọi select_test(..., final=True) ở bước đánh giá cuối.
Dev: (23922, 27)


## 6.6. Feature engineering, encoding, chuẩn hóa, chọn đặc trưng

Toàn bộ nằm trong **một sklearn Pipeline** (`src/features.py`), nên khi đánh giá chéo, pipeline được fit lại trên phần huấn luyện của từng fold:

| Bước | Kỹ thuật | Đặc trưng |
|---|---|---|
| Đặc trưng mới (feature engineering) | `row_features`: tính từ từng dòng | vị trí tương đối của đường/muối/chất béo, thành phần đứng đầu |
| Encoding biến phân loại | `OneHotEncoder` | `pnns_groups_1`, `pnns_groups_2`, NOVA (kể cả `"missing"`) |
| Encoding danh sách thẻ | multi-hot (`CountVectorizer`, `binary=True`) | thành phần, phụ gia, danh mục, nhãn, phân tích thành phần, quốc gia |
| Xử lý thiếu + chuẩn hóa | `SimpleImputer(median)` → `log(1+x)` → `StandardScaler` | số thành phần, số phụ gia, số thành phần dầu cọ, số quốc gia |
| Chọn đặc trưng | chỉ giữ thẻ có ở ≥ `min_freq` = 100 sản phẩm của tập huấn luyện | bỏ hàng chục nghìn thẻ hiếm |

**Đặc trưng mới "vị trí":** thành phần được ghi theo khối lượng giảm dần, nên đường đứng đầu danh sách nghĩa là đường chiếm nhiều nhất. `pos_sugar = 1 - vị trí / số thẻ` (1,0 nếu đứng đầu, 0 nếu không có).

In [11]:
from src.features import NUMERIC_COLUMNS, check_feature_names, feature_names, make_features, row_features

example = X[X["code"] == EXAMPLE]
print(example["product_name"].iloc[0], "|", list(example["ingredients_tags"].iloc[0])[:6])
row_features(example)[NUMERIC_COLUMNS + ["pos_sugar", "pos_salt", "pos_fat", "first_ingredient",
                                         "pnns_groups_1", "pnns_groups_2", "nova"]].T

Caramel au beurre salé | ['en:sugar', 'en:added-sugar', 'en:disaccharide', 'en:single-cream', 'en:dairy', 'en:cream']


,206
ingredients_n,5.0
additives_n,0.0
ingredients_from_palm_oil_n,0.0
n_countries,1.0
pos_sugar,1.0
pos_salt,0.083333
pos_fat,0.25
first_ingredient,[en:sugar]
pnns_groups_1,sugary snacks
pnns_groups_2,sweets


Caramel: đường đứng đầu → `pos_sugar = 1.0`; muối ở cuối → `pos_salt` nhỏ; bơ ở giữa. Giờ fit pipeline trên **phần huấn luyện của fold 0** (4/5 dev):

In [12]:
train = X_dev[split_dev["fold"] != 0]
feat = make_features().fit(train)
names = feature_names(feat)
M = feat.transform(train)
print("Ma trận:", M.shape, f"| chỉ {M.nnz / (M.shape[0] * M.shape[1]):.2%} ô khác 0 nên lưu dạng thưa (sparse)")
pd.Series([n.split("__")[0] for n in names]).value_counts().rename("số cột").to_frame().T

Ma trận: (19139, 908) | chỉ 4.45% ô khác 0 nên lưu dạng thưa (sparse)


,ingredients,categories,additives,pnns,labels,first,countries,analysis,nova,numeric,position
số cột,511,153,84,50,42,22,22,12,5,4,3


In [13]:
# Chuẩn hóa: sau StandardScaler, mỗi biến số có trung bình ~0 và độ lệch chuẩn ~1 (trên tập fit)
numeric = feat["columns"].named_transformers_["numeric"]
scaled = numeric.transform(row_features(train)[NUMERIC_COLUMNS])
pd.DataFrame(scaled, columns=NUMERIC_COLUMNS).agg(["mean", "std"]).round(3)

,ingredients_n,additives_n,ingredients_from_palm_oil_n,n_countries
mean,0.0,-0.0,0.0,-0.0
std,1.0,1.0,1.0,1.0


In [14]:
# Chọn đặc trưng bằng ngưỡng tần suất: ngưỡng thấp -> nhiều cột hơn nhiều
for mf in (20, 100):
    print(f"min_freq = {mf}: {len(feature_names(make_features(min_freq=mf).fit(train)))} đặc trưng")

min_freq = 20: 2259 đặc trưng


min_freq = 100: 908 đặc trưng


In [15]:
# Kiểm tra cuối: không đặc trưng nào là thẻ lộ đáp án hay cột dinh dưỡng
print("Tên đặc trưng sạch:", check_feature_names(names))
print("Đặc trưng có chữ 'nutriscore':", [n for n in names if "nutriscore" in n])

# Vector đặc trưng của sản phẩm ví dụ: chỉ các ô khác 0
vec = pd.Series(feat.transform(example).toarray()[0], index=names)
vec[vec != 0].round(2).to_frame("giá trị")

Tên đặc trưng sạch: True
Đặc trưng có chữ 'nutriscore': []


,giá trị
numeric__ingredients_n,-0.59
numeric__additives_n,-0.96
numeric__ingredients_from_palm_oil_n,-0.11
numeric__n_countries,-0.27
position__pos_sugar,1.00
position__pos_salt,0.08
position__pos_fat,0.25
first__en:sugar,1.00
ingredients__en:added-sugar,1.00
ingredients__en:butter,1.00


**Kết quả (Output: Clean Dataset / Modeling Dataset):**
- `data/final/clean_features.parquet` (X, 29.904 × 27) và `data/final/clean_labels.parquet` (y);
- `data/final/split.parquet` (đã khóa: dev 23.922 dòng 5 fold, test 5.982 dòng);
- pipeline đặc trưng `make_features()` khoảng 900 cột, sẽ nằm **bên trong** mô hình ở notebook 05.

Một cách chọn đặc trưng nữa (bỏ từng nhóm đặc trưng để xem nhóm nào đáng giữ) làm ở notebook 05 bằng ablation.